# Unsupervised Foundation Shade Clustering

This notebook trains K-Means on product LAB values to explore whether the
catalogue contains naturally separated colour groups. Clustering is an
optional candidate-filtering experiment; CIEDE2000 remains the final
ranking method.


## 1. Setup


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/amafteeva/Predictive-beauty-analysis-using-computer-vision-and-machine-learning-.git"


def find_project_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return None


project_root = find_project_root(Path.cwd())


    project_root = Path("/content/foundation-shade-recommender")
    if not project_root.exists():
        subprocess.run(["git", "clone", REPOSITORY_URL, str(project_root)], check=True)

if project_root is None:
    raise FileNotFoundError("Run this notebook from inside the project folder.")

os.chdir(project_root)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", "."],
    check=True,
)
print("Project root:", project_root)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from foundation_matcher.data import load_foundation_catalog
from foundation_matcher.recommender import (
    evaluate_cluster_counts,
    fit_shade_clusters,
    recommend_foundations,
    recommend_foundations_clustered,
)
from foundation_matcher.visualization import (
    plot_cluster_metrics,
    plot_shade_clusters,
)


## 2. Compare candidate values of K


In [ ]:
products = load_foundation_catalog()
cluster_evaluation = evaluate_cluster_counts(products, range(2, 13))
display(cluster_evaluation.round(3))

plot_cluster_metrics(cluster_evaluation)
plt.show()


Silhouette score is better when higher; Davies-Bouldin score is better
when lower. A mathematically strong K is not automatically the most useful
business segmentation, especially if a small K only separates light and
dark shades. Interpret cluster centres and sizes before assigning meaning.


In [ ]:
best_k = int(
    cluster_evaluation.loc[
        cluster_evaluation["silhouette_score"].idxmax(),
        "clusters",
    ]
)
print("Best K by silhouette score:", best_k)

cluster_model, clustered_products = fit_shade_clusters(products, best_k)
display(
    clustered_products["shade_cluster"]
    .value_counts()
    .sort_index()
    .rename("products")
    .to_frame()
)


In [ ]:
plot_shade_clusters(clustered_products)
plt.show()


## 3. Compare global and cluster-restricted ranking

The example below uses a synthetic LAB input so the notebook is
reproducible without a personal selfie. The clustered method predicts one
group first, then ranks products within it. Compare it with the global
baseline because cluster boundaries can exclude a genuinely close colour.


In [ ]:
example_skin_lab = np.array([65.0, 12.0, 18.0])

global_matches = recommend_foundations(
    example_skin_lab,
    products,
    top_n=5,
)
predicted_cluster, clustered_matches = recommend_foundations_clustered(
    example_skin_lab,
    clustered_products,
    cluster_model,
    top_n=5,
)

print("Predicted shade cluster:", predicted_cluster)
print("\nGlobal CIEDE2000 ranking")
display(global_matches[["brand", "product", "hex", "color_distance"]])
print("\nCluster-restricted ranking")
display(
    clustered_matches[
        ["brand", "product", "hex", "shade_cluster", "color_distance"]
    ]
)


## 4. Conclusion

Treat K-Means as an interpretable exploratory model, not evidence that it
improves recommendations. To claim improvement, compare both methods
against independent, professionally labelled foundation matches.
